# Score the detectors on DiFF, DiffusionFace, DF40, Celeb-DF and FF++

Scores the lab's photo detectors (v1, v2), the official GenD checkpoints and their fusions on the DiFF test split built by `build_diff_test.ipynb`, the `cross_diff` protocol, on the DiffusionFace sample of `build_diffusionface_test.ipynb` (`cross_diffusionface`), or on the unknown methods of DF40 built by `build_df40.ipynb` (`df40_ff`, each method scored against its own reals with `auc_within_groups`), chosen by `TEST_SET`. None of them trained on any of these. `cross_celebdf` (every crop of the official Celeb-DF v2 test videos) and `ffpp` (the official FF++ c23 test split) put the same detectors, the published ones included, on video, with the video AUC of the mean frame score. DiFF's paper and the methods compared on it report one AUC per subset (T2I, I2I, FS, FE): every pristine image against that subset's fakes, which `auc_by_group` computes.

Until the pristine images arrive the dataset holds only fakes. The notebook then reports, per subset, the mean attack score and the share of fakes scored above 0.5; no AUC exists without the bona fide side.

**Fair comparison:** published DiFF numbers come from detectors trained on FaceForensics++ alone. v1 and v2 trained on DeepFakeFace, which holds Stable Diffusion faces, so their DiFF numbers are not comparable to those without saying so. On DiFF, Celeb-DF and FF++ the notebook also scores the published FSFM and FS-VFM checkpoints fine-tuned on FF++ DeepFakes c23 (`Wolowolo/fsfm-3c`, CC BY-NC 4.0, the rows of Table V in arXiv 2510.10663) and the lab's recipe trained under `ffpp_deepfakes_to_diff`, on the same crops. Their thresholds for detection at BPCER 0.1%, 0.2% and 0.5% are all set on the same bona fide crops, the FF++ validation originals (`detection_at_bpcer`).

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secret `HF_TOKEN`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
TEST_SET = "ffpp"  # or "cross_diff", "cross_diffusionface", "df40_ff" or "ffpp"
# Datasets each test set loads; DiFF keeps its pristine images in a dataset of their own.
TEST_DATASETS = {"cross_diff": ("paulohenriquevn/face-decision-diff-test", "paulohenriquevn/face-decision-diff-pristine-test"),
                 "cross_diffusionface": ("paulohenriquevn/face-decision-diffusionface-test",),
                 "df40_ff": ("paulohenriquevn/face-decision-df40",),
                 "cross_celebdf": ("paulohenriquevn/face-decision-celebdf-test",),
                 "ffpp": ("paulohenriquevn/face-decision-ffpp",)}
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # reports go to HF_MODEL/pretrained/<detector>/<TEST_SET>.json
CLIP_MODEL = "openai/clip-vit-large-patch14"
GEND_MODELS = {"clip": "yermandy/GenD_CLIP_L_14", "pe": "yermandy/GenD_PE_L"}
GEND_FACE_SCALE = 1.3
# Photo experts: v1 (subject_holdout checkpoint of commit b66500a) and v2 (lab_ffpp_real, commit b962286).
V1_CHECKPOINT = "subject_holdout/trained_parameters.pt"
V2_CHECKPOINT = "openai-ce-e1-b32-lr0.0003-fs1.5-g10-noaug/lab_ffpp_real/checkpoint.pt"
# The lab's recipe trained on FF++ originals and DeepFakes alone (commit 8edf794), the training data
# of the published DiFF rows.
FFPP_DEEPFAKES_CHECKPOINT = "openai-ce-e1-b32-lr0.0003-fs1.5-g10-noaug/ffpp_deepfakes_to_diff/checkpoint.pt"
# Published detectors fine-tuned on FF++ DeepFakes c23: folder and ViT size. Scored with the lab's
# recipe on the same data on every test set that has a calibration side (CALIBRATED_TEST_SETS).
FSFM_REPO = "Wolowolo/fsfm-3c"
FSFM_MODELS = {
    "fsvfm_vit_l": ("finetuned_models/FS-VFM_extensions/cross_dataset_DFD_and_DiFF/ViT-L_VF2_600e/FT_on_FF++_DF_c23_32frames", "L"),
    "fsvfm_vit_b": ("finetuned_models/FS-VFM_extensions/cross_dataset_DFD_and_DiFF/ViT-B_VF2_600e/FT_on_FF++_DF_c23_32frames", "B"),
    "fsfm_vit_b": ("finetuned_models/FF++_DF_c23_32frames/correct_version", "B"),
}
# FSFM crops a square 1.3 times the dlib box; tighten_crop reframes the lab's 1.5x RetinaFace crops.
FSFM_FACE_SCALE = 1.3
# Bona fide crops every threshold is set on: the FF++ validation originals.
CALIBRATED_TEST_SETS = ("cross_diff", "cross_celebdf", "ffpp")
CALIBRATION_DATASET = "paulohenriquevn/face-decision-ffpp"


In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" "timm>=1.0.20"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Load the test set

In [ ]:
from face_decision_lab.hub import load_hub_samples
from face_decision_lab.protocol import protocol_split

samples, images = [], {}
for index, repo in enumerate(TEST_DATASETS[TEST_SET]):
    part, part_images = load_hub_samples(repo, key_prefix = f"{TEST_SET}{index}")
    images.update(zip([s.image_path for s in part], part_images, strict = True))
    samples += part
# DiFF, DiffusionFace and Celeb-DF test on every crop; DF40 on its unknown methods, FF++ on its
# official test split. Nothing here trains, so the lab data is not loaded.
test = protocol_split(samples, TEST_SET)[2] if TEST_SET in ("df40_ff", "ffpp") else samples
bona_fide = sum(s.is_bona_fide for s in test)
print(len(test), "crops,", bona_fide, "bona fide")
calibration = []
if TEST_SET in CALIBRATED_TEST_SETS:
    if TEST_SET == "ffpp":
        ffpp = samples
    else:
        ffpp, ffpp_images = load_hub_samples(CALIBRATION_DATASET, key_prefix = "faceforensics")
        images.update(zip([s.image_path for s in ffpp], ffpp_images, strict = True))
    calibration = [s for s in protocol_split(ffpp, "ffpp")[1] if s.is_bona_fide]
    print(len(calibration), "bona fide calibration crops")

### Detectors

In [ ]:
import importlib.util, io, json
from functools import partial
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from timm.models.vision_transformer import VisionTransformer
from torchvision import transforms
from safetensors.torch import load_file
from torch import nn
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel, CLIPVisionModelWithProjection
from face_decision_lab.faces import tighten_crop
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
EXAMPLES = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
example_frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in EXAMPLES]

def load_gend(repo):
    spec = importlib.util.spec_from_file_location(f"modeling_{repo.split('/')[1]}", hf_hub_download(repo, "modeling_gend.py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    model = module.GenD(module.GenDConfig.from_pretrained(repo))
    loaded = model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors")), strict = False)
    assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
    model = model.cuda().eval()
    with torch.no_grad():
        fake = model(torch.stack([model.feature_extractor.preprocess(f) for f in example_frames]).cuda()).softmax(-1)[:, 1].tolist()
    assert fake[0] > 0.5 > fake[1], (repo, fake)
    return model

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

class LayerNormTunedCLIPv1(nn.Module):  # the v1 detector, as trained at commit b66500a
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        return torch.sigmoid(self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)).squeeze(-1))

class LayerNormTunedCLIP(nn.Module):  # clip_layernorm_detector.ipynb's detector (pooled features, two-class head)
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, pixels):
        return self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).pooler_output, dim = -1)).softmax(-1)[:, 1]

def load_v1():
    model = LayerNormTunedCLIPv1()
    model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, V1_CHECKPOINT)), strict = False)
    return model.cuda().eval()

def load_layernorm_clip(checkpoint):
    model = LayerNormTunedCLIP()
    loaded = model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, checkpoint), weights_only = False)["trainable"], strict = False)
    assert not loaded.unexpected_keys, loaded.unexpected_keys[:5]
    return model.cuda().eval()

def load_fsfm(folder, size):
    # FSFM's models_vit: timm's ViT/16 averaging the patch tokens without the class token, then fc_norm.
    shape = {"B": dict(embed_dim = 768, depth = 12, num_heads = 12), "L": dict(embed_dim = 1024, depth = 24, num_heads = 16)}[size]
    model = VisionTransformer(img_size = 224, patch_size = 16, mlp_ratio = 4, qkv_bias = True, num_classes = 2, global_pool = "avg",
                              fc_norm = True, norm_layer = partial(nn.LayerNorm, eps = 1e-6), **shape)
    state = torch.load(hf_hub_download(FSFM_REPO, f"{folder}/checkpoint-min_val_loss.pth"), map_location = "cpu", weights_only = False)["model"]
    model.load_state_dict(state, strict = True)
    with open(hf_hub_download(FSFM_REPO, f"{folder}/pretrain_ds_mean_std.txt")) as f:
        stats = json.loads(f.readline())
    to_input = transforms.Compose([transforms.Resize((224, 224), interpolation = transforms.InterpolationMode.BICUBIC),
                                   transforms.ToTensor(), transforms.Normalize(stats["mean"], stats["std"])])
    model = model.cuda().eval()
    # FSFM's ImageFolder sorts fake_* before real_*, so class 0 is fake.
    fake = lambda x: model(x).softmax(-1)[:, 0]
    with torch.no_grad():
        example = fake(torch.stack([to_input(f) for f in example_frames]).cuda()).tolist()
    assert example[0] > example[1], (folder, example)
    return to_input, fake

def scores_of(part, prepare, model):
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            out += model(torch.stack(list(pool.map(prepare, part[i:i + 128]))).cuda()).float().cpu().tolist()
    return np.array(out)

def clip_pixels(sample):  # the lab detectors trained on the 1.5x crops as built
    return processor(images = open_image(images[sample.image_path]), return_tensors = "pt")["pixel_values"][0]

scores, calibration_scores = {}, {}

def score_both(name, prepare, model):  # the test side and, on DiFF, the calibration reals
    scores[name] = scores_of(test, prepare, model)
    if calibration:
        calibration_scores[name] = scores_of(calibration, prepare, model)

def fuse(name, *parts, how):
    scores[name] = how(*(scores[p] for p in parts))
    if calibration:
        calibration_scores[name] = how(*(calibration_scores[p] for p in parts))

for name, repo in GEND_MODELS.items():
    model = load_gend(repo)

    def prepare(s, m = model):
        return m.feature_extractor.preprocess(tighten_crop(open_image(images[s.image_path]), GEND_FACE_SCALE))

    score_both(f"gend_{name}", prepare, lambda x, m = model: m(x).softmax(-1)[:, 1])
    del model
fuse("gend_ensemble_clip_pe", "gend_clip", "gend_pe", how = lambda a, b: (a + b) / 2)
score_both("layernorm_clip_v1", clip_pixels, load_v1())
score_both("v2_ffpp_real", clip_pixels, load_layernorm_clip(V2_CHECKPOINT))
fuse("fusion_max_v1_gendens", "layernorm_clip_v1", "gend_ensemble_clip_pe", how = np.maximum)
fuse("fusion_max_v2_ffpp_real_gendens", "v2_ffpp_real", "gend_ensemble_clip_pe", how = np.maximum)
if TEST_SET in CALIBRATED_TEST_SETS:
    score_both("lab_ffpp_deepfakes", clip_pixels, load_layernorm_clip(FFPP_DEEPFAKES_CHECKPOINT))
    for name, (folder, size) in FSFM_MODELS.items():
        to_input, fake = load_fsfm(folder, size)
        score_both(name, lambda s, t = to_input: t(tighten_crop(open_image(images[s.image_path]), FSFM_FACE_SCALE)), fake)
        del fake
        torch.cuda.empty_cache()

### Reports

With bona fide crops: the full evaluation report plus the AUC of each DiFF subset (on DiffusionFace, `auc_by_generator` gives one AUC per folder). Without them: per subset, the mean attack score and the share of fakes above 0.5. On DiFF, Celeb-DF and FF++, each report adds `detection_at_bpcer`, with every threshold set on the FF++ validation originals. Every crop's score goes to `HF_MODEL/scores/<TEST_SET>.npz` (`score_files.save_scores`), and the calibration crops' scores to `HF_MODEL/scores/<TEST_SET>_calibration.npz`.

In [ ]:
from face_decision_lab.protocol import auc_by_group, auc_within_groups, detection_at_bpcer, evaluation_report
from face_decision_lab.score_files import save_scores
from face_decision_lab.sources import df40, diff

KEYS = ("samples", "auc", "eer", "bpcer_at_max_apcer", "auc_by_generator", "videos", "video_auc")
# DiFF groups its methods into four subsets; DiffusionFace reports each folder on its own.
group = diff.subset if TEST_SET == "cross_diff" else (lambda generator: generator)
subsets = np.array([None if s.is_bona_fide else group(s.generator) for s in test])
names = sorted(set(subsets[subsets != None]))  # noqa: E711

def summary(values):
    if bona_fide:
        result = {k: v for k, v in evaluation_report(test, values.tolist()).items() if k in KEYS}
        # DF40's unknown methods each ship their own reals; DiFF and DiffusionFace share one set.
        if TEST_SET == "df40_ff":
            result["auc_by_subset"] = auc_within_groups(test, values.tolist(), df40.unknown_group)
        else:
            result["auc_by_subset"] = auc_by_group(test, values.tolist(), group = group)
        result["mean_auc_over_subsets"] = float(np.mean(list(result["auc_by_subset"].values())))
    else:
        result = {"samples": len(test), "bona_fide": 0}
    result["mean_score_by_subset"] = {k: float(values[subsets == k].mean()) for k in names}
    result["share_above_half_by_subset"] = {k: float((values[subsets == k] > 0.5).mean()) for k in names}
    return result

for detector, values in scores.items():
    result = summary(values)
    if calibration:
        result["detection_at_bpcer"] = detection_at_bpcer(calibration, calibration_scores[detector].tolist(), test, values.tolist())
    api.upload_file(path_or_fileobj = json.dumps(result, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = f"pretrained/{detector}/{TEST_SET}.json", commit_message = f"{detector} on {TEST_SET}")
    shown = "auc_by_subset" if bona_fide else "share_above_half_by_subset"
    print(f"{detector:34s}", {k: round(v, 4) for k, v in result[shown].items()})
# Every crop's score, for intervals and comparisons computed later without the GPU.
save_scores(f"/content/{TEST_SET}_scores.npz", test, scores)
api.upload_file(path_or_fileobj = f"/content/{TEST_SET}_scores.npz", repo_id = HF_MODEL,
                path_in_repo = f"scores/{TEST_SET}.npz", commit_message = f"Per-crop scores on {TEST_SET}")
if calibration:
    save_scores(f"/content/{TEST_SET}_calibration_scores.npz", calibration, calibration_scores)
    api.upload_file(path_or_fileobj = f"/content/{TEST_SET}_calibration_scores.npz", repo_id = HF_MODEL,
                    path_in_repo = f"scores/{TEST_SET}_calibration.npz", commit_message = f"Per-crop calibration scores for {TEST_SET}")
